# 데이터 연결 기준

온도 자료의 설비, LOT 관측 구간, 강종과 이벤트를 연결함.


In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display



FIG_DIR = os.path.join(ROOT, 'figures', 'namelesspark', '01_data_alignment')
CAL_TAG = 'P1-CR1-CAL01'
PLANT_CD = 'CR1'

# 설치된 한글 폰트를 사용함.
font_names = []

for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)

for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if font_name in font_names:
        rcParams['font.family'] = font_name
        break
else:
    print('경고: 사용할 수 있는 한글 폰트를 찾지 못했음. 그래프의 한글이 깨질 수 있음.')

rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110

# CSV를 직접 읽고, 계산에 사용하는 날짜 컬럼을 날짜형으로 바꿈.
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

temp['MEAS_DT'] = pd.to_datetime(temp['MEAS_DT'], errors='raise')
event['EVT_DT'] = pd.to_datetime(event['EVT_DT'], errors='raise')
repair['STRT_DT'] = pd.to_datetime(repair['STRT_DT'], errors='raise')
repair['END_DT'] = pd.to_datetime(repair['END_DT'], errors='raise')
eqp['INSTALL_DT'] = pd.to_datetime(eqp['INSTALL_DT'], errors='raise')
maint['DETC_DT'] = pd.to_datetime(maint['DETC_DT'], errors='raise')
maint['STRT_DT'] = pd.to_datetime(maint['STRT_DT'], errors='raise')
maint['END_DT'] = pd.to_datetime(maint['END_DT'], errors='raise')

# 읽은 원본 데이터프레임을 이름별로 보관함.
sources = {
    'temp': temp,
    'event': event,
    'repair': repair,
    'eqp': eqp,
    'part': part,
    'maint': maint,
    'partchg': partchg,
}

print("온도 행 수:", len(temp))

## 1. 온도 자료의 설비

파일명의 `CR1`, `CAL01`로 설비마스터에서 `P1-CR1-CAL01`을 찾음. 온도 자료 전체가 같은 설비이므로 동일한 설비 정보를 붙임.


In [ ]:
target_eqp = eqp.loc[eqp["EQP_TAG"].eq(CAL_TAG)].copy()

# 센서 전체 값 대신 설비 연결을 확인할 시간, LOT만 사용함.
joined_keys = temp[["MEAS_DT", "LOT_NO"]].assign(EQP_TAG=CAL_TAG).merge(
    target_eqp, on="EQP_TAG", how="left", validate="m:1", indicator=True
)

install_dt = pd.to_datetime(target_eqp["INSTALL_DT"].iloc[0])

installed_before_measurement = joined_keys["MEAS_DT"].ge(install_dt)

display(pd.DataFrame({
    "확인 항목": ["대상 설비", "대상 설비 행 수", "조인 전 행 수", "조인 후 행 수",
                 "설비 정보 미연결 행 수", "설치일보다 앞선 측정 행 수"],
    "결과": [CAL_TAG, len(target_eqp), len(temp), len(joined_keys),
             int(joined_keys["_merge"].ne("both").sum()),
             int((~installed_before_measurement).sum())],
}))

del joined_keys

`EQP_TAG`는 정비이력에서 같은 설비를 찾는 기준임. 설비마스터에는 변경 이력이 없으며, 정비 여러 건을 온도에 바로 붙이면 센서 행이 중복될 수 있음.


## 2. LOT의 실제 관측 구간

LOT별 첫 `MEAS_DT`부터 마지막 `MEAS_DT`까지 사용하고 관측 공백은 채우지 않음. 시작 시각의 `CR1` 강종변경에서 변경 후 강종(`AFT_VAL`)을 연결함.

`DURATION_H`는 첫 관측부터 마지막 관측까지의 시간이며 실제 가동시간은 아님.


In [ ]:
# LOT의 실제 첫 관측과 마지막 관측을 구함.

lot_bounds = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"),
    END=("MEAS_DT", "max"),
    ROWS=("MEAS_DT", "size"),
)

# CR1 강종변경 중 LOT 첫 관측과 시각이 정확히 같은 기록만 연결함.
lot_grade_changes = event.loc[
    event["PLANT_CD"].eq("CR1") & event["EVT_TYPE"].eq("강종변경"),
    ["EVT_DT", "BEF_VAL", "AFT_VAL"],
].sort_values("EVT_DT")

lot_bounds = lot_bounds.merge(
    lot_grade_changes[["EVT_DT", "AFT_VAL"]],
    left_on="START", right_on="EVT_DT", how="left", validate="many_to_one",
)

lot_bounds = lot_bounds.rename(columns={"AFT_VAL": "GRADE"}).drop(columns="EVT_DT")



In [ ]:
# LOT별 관측 기간과 강종을 정리함.

lot_bounds["DURATION_H"] = (lot_bounds["END"] - lot_bounds["START"]).dt.total_seconds() / 3600

lot_context = lot_bounds[["LOT_NO", "GRADE", "START", "END", "ROWS", "DURATION_H"]]

lot_context = lot_context.sort_values("START").reset_index(drop=True)


In [ ]:
# LOT 연결 결과와 CR1 강종변경 기록을 정리함.

cr1_event = event.loc[event["PLANT_CD"].eq(PLANT_CD)].copy()

cr1_grade = cr1_event.loc[cr1_event["EVT_TYPE"].eq("강종변경")].sort_values("EVT_DT")

display(lot_context)


### LOT와 강종이 맞는지 확인

관측 중 추가 강종변경이 있는지, LOT 종료 후 첫 변경의 이전 강종(`BEF_VAL`)이 같은지 확인함. 이벤트 시각이 실제 소재 투입 시각인지는 별도 확인이 필요함.


In [ ]:
grade_checks = []

for row in lot_context.itertuples(index=False):
    inside = cr1_grade.loc[cr1_grade["EVT_DT"].between(row.START, row.END)]

    later = cr1_grade.loc[cr1_grade["EVT_DT"].gt(row.END)]

    next_before = later["BEF_VAL"].iloc[0] if len(later) else pd.NA

    grade_checks.append({
        "LOT_NO": row.LOT_NO,
        "GRADE": row.GRADE,
        "관측구간_강종변경수": len(inside),
        "시작시각_강종변경수": int(inside["EVT_DT"].eq(row.START).sum()),
        "다음변경_BEF_VAL": next_before,
        "다음변경과_일치": bool(next_before == row.GRADE) if pd.notna(next_before) else pd.NA,
    })

grade_check = pd.DataFrame(grade_checks)

display(grade_check)

# 전체 이벤트의 강종 수와 실제 온도가 관측된 강종 수를 구분함.
observed_grades = set(lot_context["GRADE"])
event_grades = set(cr1_grade["BEF_VAL"].dropna()) | set(cr1_grade["AFT_VAL"].dropna())

display(lot_context.groupby("GRADE").agg(LOT수=("LOT_NO", "size"), 온도행수=("ROWS", "sum")))
print("CR1 이벤트에는 있지만 온도 LOT에는 없는 강종:", sorted(event_grades - observed_grades))

## 3. 온도 관측과 이벤트가 겹치는 기간

`CR1`의 전체 날짜 범위와 실제 LOT 관측 구간을 나누어 셈. 첫날과 마지막 날 사이에 온도가 계속 기록된 것은 아님.


In [ ]:
cr1_repair = repair.loc[repair["PLANT_CD"].eq(PLANT_CD)].copy()

measure_start, measure_end = temp["MEAS_DT"].min(), temp["MEAS_DT"].max()

display(pd.DataFrame([
    {"파일": "온도", "대상 행 수": len(temp), "시작": measure_start, "끝": measure_end},
    {"파일": "CR1 조업이벤트", "대상 행 수": len(cr1_event),
     "시작": cr1_event["EVT_DT"].min(), "끝": cr1_event["EVT_DT"].max()},
    {"파일": "CR1 정기수리", "대상 행 수": len(cr1_repair),
     "시작": cr1_repair["STRT_DT"].min(), "끝": cr1_repair["END_DT"].max()},
]))

in_observation = pd.Series(False, index=cr1_event.index)

for row in lot_context.itertuples(index=False):
    in_observation |= cr1_event["EVT_DT"].between(row.START, row.END)

in_calendar = cr1_event["EVT_DT"].between(measure_start, measure_end)

event_overlap = pd.DataFrame({
    "구분": ["CR1 이벤트 전체", "온도 전체 달력 범위 안", "실제 LOT 관측 구간 안",
             "실제 LOT 관측 구간 밖"],
    "이벤트 수": [len(cr1_event), int(in_calendar.sum()), int(in_observation.sum()),
                 int((~in_observation).sum())],
})

display(event_overlap)

### 정기수리 중 온도 관측 여부

정기수리 종료일을 포함할 때와 제외할 때를 비교함. 날짜로 된 수리 계획만으로 실제 작업 종료 시각은 알 수 없음.


In [ ]:
repair_checks = []

for row in cr1_repair.itertuples(index=False):
    raw_count = temp["MEAS_DT"].between(row.STRT_DT, row.END_DT).sum()
    end_day_exclusive = row.END_DT.normalize() + pd.Timedelta(days=1)
    inclusive_day_count = (temp["MEAS_DT"].ge(row.STRT_DT) &
                           temp["MEAS_DT"].lt(end_day_exclusive)).sum()
    repair_checks.append({
        "PLAN_ID": row.PLAN_ID,
        "STRT_DT": row.STRT_DT,
        "END_DT": row.END_DT,
        "CSV종료시각_기준_온도행수": int(raw_count),
        "종료일포함_기준_온도행수": int(inclusive_day_count),
    })

repair_overlap = pd.DataFrame(repair_checks)

display(repair_overlap)

### 자료가 있는 기간

온도 LOT 사이의 빈 구간은 관측값이 없는 기간임.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.2))

for row in lot_context.itertuples(index=False):
    ax.plot([row.START, row.END], [2, 2], color="#2a78d6", linewidth=7, solid_capstyle="butt")

ax.scatter(cr1_grade["EVT_DT"], [1] * len(cr1_grade), marker="|", color="#65758b", s=70)

for row in cr1_repair.itertuples(index=False):
    ax.plot([row.STRT_DT, row.END_DT], [0, 0], color="#eb6834", linewidth=8, solid_capstyle="butt")

ax.set_yticks([0, 1, 2], ["CR1 정기수리", "CR1 강종변경", "실제 온도 LOT 관측"])
ax.set_ylim(-0.5, 2.5)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_title("전체 달력 범위와 실제 온도 관측 구간은 다름")
ax.grid(axis="x", color="#dde3eb", linewidth=0.7)
fig.autofmt_xdate()
fig.tight_layout()

os.makedirs(FIG_DIR, exist_ok=True)

fig.savefig(os.path.join(FIG_DIR, 'observation_event_repair_coverage.png'), dpi=150, bbox_inches="tight")

plt.show()

## 연결 결과

| 항목 | 결과 |
| --- | --- |
| 설비 | `P1-CR1-CAL01` |
| LOT와 강종 | 14개 LOT, 7개 강종. 각 LOT 관측 중 강종은 하나임 |
| 관측 구간 안의 CR1 이벤트 | 전체 134건 중 14건 |
| CR1 정기수리 2회 | 두 기간 모두 온도 관측 없음. 종료일 포함 여부와 무관함 |

`SPCC`는 이벤트에는 있지만 온도 자료에는 없음. LOT 밖의 이벤트는 오류가 아니라 해당 시각의 온도 관측값이 없는 것임.
